# 🔍 Cross-Domain EDA and Vector Indexing
This notebook performs Exploratory Data Analysis on Hazards, Permits, and Waste datasets before building the 30k RAG index.

## 🚧 1. 311 Service Requests (Hazards)
Visualizing the most common types of city-wide hazards.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import json
import sys

# UPDATED: Use a relative path to the project root
# Since this notebook is in the 'notebooks' folder, we go up one level to reach 'data'
data_dir = "../data"

# --- 1. 311 Hazards EDA ---
if os.path.exists(os.path.join(data_dir, "311_service_requests.csv")):
    df_311 = pd.read_csv(os.path.join(data_dir, "311_service_requests.csv"), encoding="latin1", low_memory=False)
    df_311['Service Request Type'].value_counts().head(10).plot(kind='bar', color='orange', title='Top 10 City Hazards')
    plt.ylabel('Count')
    plt.show()
else:
    print("⚠️ 311 CSV not found. Check your 'data' folder.")

## 🏗️ 2. Building Permits
Analyzing the distribution of construction work types across Toronto.

In [ ]:
# --- 2. Building Permits EDA ---
permits_path = os.path.join(data_dir, "Cleared Building Permits since 2017.json")
if os.path.exists(permits_path):
    with open(permits_path, 'r') as f:
        permits = json.load(f)
    df_permits = pd.DataFrame(permits)
    df_permits['WORK'].value_counts().head(8).plot(kind='pie', autopct='%1.1f%%', title='Building Permit Work Distribution')
    plt.show()

## ♻️ 3. Waste Wizard
Understanding the categorization of household waste items.

In [ ]:
# --- 3. Waste Wizard EDA ---
waste_path = os.path.join(data_dir, "Waste Wizard Lookup Table.json")
if os.path.exists(waste_path):
    with open(waste_path, 'r') as f:
        waste = json.load(f)
    df_waste = pd.DataFrame(waste)
    df_waste['category'].value_counts().plot(kind='barh', color='green', title='Waste Item Categories')
    plt.xlabel('Number of Unique Items')
    plt.show()

## 🏗️ Execute 30k RAG Indexing
Running the master ingestion script to build the vector database.

In [ ]:
# --- 4. Indexing Execution ---
# UPDATED: Add the parent directory (root) to the path so we can import from 'modules'
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

try:
    from modules.ingestor import build_30k_rag
    # Run the indexing process using the current directory as the project path
    vectorstore = build_30k_rag(project_path="..")
except ImportError as e:
    print(f"❌ Could not import ingestor. Make sure 'modules/ingestor.py' exists. Error: {e}")